# Actividad 6 | Evaluación y validación de modelos de clasificación bancaria

**Origen:** resultados verificables de la Actividad 5. Se comparan Regresión Logística y Multinomial Naive Bayes en 77 intenciones.

**Objetivo SMART:** evaluar ambos modelos sobre el conjunto de prueba, contrastar F1 macro con la meta de 0.90, estudiar cobertura y errores según umbral de confianza, y documentar una propuesta de prueba A/B operativa.

> Nota metodológica: un análisis retrospectivo de predicciones no es una prueba A/B desplegada. La curva ROC/AUC multiclase ya fue calculada en la Actividad 5 y se reporta como resultado histórico; para reconstruir la curva completa se requieren las probabilidades por clase o volver a ejecutar los modelos.

## 1. Cargar resultados
Sube `Actividad5_ejecutada.zip` a Colab (o colócalo junto al notebook). No requiere reentrenar modelos.

In [ ]:
import io, zipfile, pathlib, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
from statsmodels.stats.contingency_tables import mcnemar
zip_path = pathlib.Path('Actividad5_ejecutada.zip')
if not zip_path.exists():
    from google.colab import files
    files.upload()
assert zip_path.exists(), 'Sube Actividad5_ejecutada.zip'
z = zipfile.ZipFile(zip_path)
root = 'Actividad5/resultados/'
def csv(name): return pd.read_csv(io.BytesIO(z.read(root + name)))
def js(name): return json.loads(z.read(root + name).decode('utf-8'))
metrics = csv('metricas.csv')
log = csv('predicciones_Regresion_logistica.csv')
nb = csv('predicciones_Multinomial_NB.csv')
print('Muestras:', len(log), 'Clases:', log.category.nunique())
display(metrics)


## 2. Métricas y comparación con baseline

In [ ]:
baseline = js('baseline.json')
print('Baseline registrado:', baseline)
cols = ['accuracy','precision_macro','recall_macro','f1_macro','roc_auc_ovr_macro']
ax = metrics.set_index('modelo')[cols].T.plot(kind='bar', figsize=(10,4), ylim=(0,1.08), rot=25)
ax.set_ylabel('Valor'); ax.set_title('Métricas en test (ROC-AUC multiclase OvR)')
plt.tight_layout(); plt.show()
print('Meta F1 macro >= 0.90:', dict(zip(metrics.modelo,metrics.f1_macro >= .90)))


## 3. Matriz de confusión y errores frecuentes
La matriz de 77 × 77 se resume visualmente para evitar etiquetas ilegibles; se identifican confusiones específicas.

In [ ]:
for name, df in [('Regresión Logística',log),('Naive Bayes',nb)]:
    labels = sorted(df.category.unique())
    cm = confusion_matrix(df.category, df.prediccion, labels=labels)
    fig,ax=plt.subplots(figsize=(7,6)); ax.imshow(cm,aspect='auto',cmap='Blues')
    ax.set(title='Matriz de confusión: '+name,xlabel='Predicción',ylabel='Etiqueta real')
    plt.tight_layout();plt.show()
    errs=(df.loc[df.category != df.prediccion].groupby(['category','prediccion']).size().sort_values(ascending=False).head(10))
    print(name,'Errores:',int((df.category!=df.prediccion).sum())); display(errs.rename('casos').reset_index())


## 4. Umbrales de confianza y derivación a revisión humana
`score_maximo` es la confianza de la clase elegida. Se evalúa una **política de abstención**: aceptar automáticamente predicciones con confianza ≥ umbral; enviar el resto a revisión. La precisión de los casos aceptados no es una métrica de clasificación binaria estándar.

In [ ]:
rows=[]
for name,df in [('Regresión Logística',log),('Naive Bayes',nb)]:
    for t in [0.0,0.3,0.5,0.7,0.8,0.9]:
        accepted=df.score_maximo>=t
        rows.append({'modelo':name,'umbral':t,'cobertura':accepted.mean(),
                     'exactitud_aceptados':df.loc[accepted,'correcto'].mean() if accepted.any() else np.nan,
                     'a_revision':int((~accepted).sum()),'errores_automaticos':int((accepted & ~df.correcto.astype(bool)).sum())})
thresholds=pd.DataFrame(rows); display(thresholds)
for name,g in thresholds.groupby('modelo'):
    plt.plot(g.cobertura,g.exactitud_aceptados,marker='o',label=name)
plt.xlabel('Cobertura automática');plt.ylabel('Exactitud en casos aceptados');plt.title('Compromiso cobertura–exactitud');plt.legend();plt.grid(alpha=.2);plt.show()


## 5. Validación cruzada de cinco particiones

In [ ]:
cv=metrics[['modelo','cv_f1_macro','cv_f1_std','f1_macro']].copy()
cv['brecha_test_menos_cv']=cv.f1_macro-cv.cv_f1_macro
display(cv)
print('La desviación CV resume variabilidad entre folds; no es por sí sola un intervalo de confianza.')


## 6. Comparación pareada y significancia
McNemar compara aciertos y errores de ambos modelos sobre **las mismas observaciones**. No equivale a una prueba A/B con usuarios asignados aleatoriamente.

In [ ]:
paired=log[['id','category','correcto']].merge(nb[['id','category','correcto']],on=['id','category'],suffixes=('_lr','_nb'),validate='one_to_one')
a=paired.correcto_lr.astype(bool);b=paired.correcto_nb.astype(bool)
table=np.array([[int((~a&~b).sum()),int((~a&b).sum())],[int((a&~b).sum()),int((a&b).sum())]])
print('Filas: LR falla/acierta; columnas: NB falla/acierta')
print(table)
r=mcnemar(table,exact=False,correction=True)
print('McNemar p:',r.pvalue)
print('Interpretación:', 'evidencia de diferencia en tasa de aciertos' if r.pvalue<.05 else 'evidencia insuficiente de diferencia')


## 7. Prueba A/B propuesta (NO ejecutada)
**Hipótesis:** en tráfico bancario real, Regresión Logística reduce las derivaciones incorrectas frente a Naive Bayes. **Unidad de aleatorización:** conversación/usuario según control de contaminación. **A:** Naive Bayes; **B:** Regresión Logística. **Métrica primaria:** proporción de intenciones correctamente enrutadas según revisión humana independiente. **Guardrails:** latencia, tasa de revisión, quejas y tasa de fallos críticos. **Diseño:** asignación aleatoria, tamaño de muestra y duración predefinidos, monitoreo de sesgos y consentimiento/privacidad aplicables. **Análisis:** contraste de dos proporciones con intervalo de confianza; ajustar por múltiples pruebas si procede. No se reportan resultados ficticios de un experimento en producción.

## 8. Conclusiones y trazabilidad
Los resultados históricos favorecen Regresión Logística en F1 macro y exactitud. La meta F1 macro 0.90 debe verificarse con el CSV, no redondearse de manera engañosa. Los umbrales de confianza ofrecen una decisión de operación entre automatización y revisión humana. Para construir **curvas ROC reales**, exportar la matriz de probabilidades de las 77 clases desde el pipeline de la Actividad 5 y recalcular OvR; un único `score_maximo` por observación no basta.

In [ ]:
print(metrics[['modelo','f1_macro','roc_auc_ovr_macro']].to_string(index=False))
print('Comparación estadística histórica:'); print(js('comparacion_estadistica.json'))
